In [8]:
import numpy as np
import tensorflow as tf

2025-11-13 18:46:00.710646: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-13 18:46:00.714220: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /apps/software/2024a/software/code-server/4.103.0/lib:/apps/software/2024a/software/gettext/0.22.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libiconv/1.17-GCCcore-13.3.0/lib:/apps/software/2024a/software/ncurses/6.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/apps/software/2024a/software/XZ/5.4.5-GCCcore-13.3.0/lib:/apps/software/2024a/software/expat/2.6.2-GCCcore-13.3.0/lib:/apps/software/2024a/software/cUR

In [2]:
data_root="/home/mcn26/project_pi_skr2/shared/tabula_data/simulated/dumps"

In [5]:
def load(file):
    return np.load(f"{data_root}/{file}")
endog=load("52936bcb-c2ff-4120-aef8-4b29d04bb5d4_endog.npy")
exog=load("52936bcb-c2ff-4120-aef8-4b29d04bb5d4_exog.npy")
G=load("52936bcb-c2ff-4120-aef8-4b29d04bb5d4_G.npy")
infl=load("52936bcb-c2ff-4120-aef8-4b29d04bb5d4_infl.npy")
params=load("52936bcb-c2ff-4120-aef8-4b29d04bb5d4_params.npy")

In [9]:
def _zinb_loglik_tf(params, exog, exog_infl, endog):
    """
    TF implementation of the ZINB log-likelihood works for arbitrary exog / exog_infl shapes.

    params = concat([x_mu, x_pi, log_theta])
    """
    N = tf.cast(tf.shape(endog)[0], tf.float64)

    num_features = tf.shape(exog)[1]
    num_infl_features = tf.shape(exog_infl)[1]

    x_mu = params[:num_features]
    x_pi = params[num_features:num_features + num_infl_features]
    log_theta = params[-1]
    theta = tf.exp(log_theta)

    mu = tf.exp(tf.matmul(exog, tf.expand_dims(x_mu, axis=-1)))
    pi_logits = tf.matmul(exog_infl, tf.expand_dims(x_pi, axis=-1))

    # zero-inflation logits -> log(q0), log(q1) 
    log_q0 = -tf.nn.softplus(-pi_logits)
    log_q1 = log_q0 - pi_logits

    y = tf.cast(endog, tf.float64)

    # NB log-likelihood (for y>0)
    t1 = tf.math.lgamma(y + theta)
    t2 = -tf.math.lgamma(theta)
    t3 = theta * log_theta
    t4 = y * tf.math.log(mu + 1e-8)
    ty = tf.math.log(mu + theta + 1e-8)
    t5 = -(theta + y) * ty
    nb_case = t1 + t2 + t3 + t4 + t5 + log_q1

    # Zero case
    p1 = theta * (log_theta - ty) + log_q1
    zero_case = tf.reduce_logsumexp(tf.stack([log_q0, p1], axis=0), axis=0)

    ll = tf.where(y < 1e-8, zero_case, nb_case)

    # 3-step reduction (mean neg LL per output, add back log-factorial, sum)
    mean_neg_ll = -tf.reduce_mean(ll, axis=0)                      # (num_outputs,)
    log_fact = tf.reduce_sum(tf.math.lgamma(endog + 1), axis=0)    # ∑ ln(y!)
    llfs = -(mean_neg_ll * N + log_fact)                           # (num_outputs,)
    log_likelihood = tf.reduce_sum(llfs)
    return log_likelihood

In [10]:
_zinb_loglik_tf(params, exog, exog_infl=infl, endog=endog)

2025-11-13 18:46:06.673038: E tensorflow/stream_executor/cuda/cuda_driver.cc:271] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-11-13 18:46:06.673073: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:156] kernel driver does not appear to be running on this host (a1130u11n03.mghpcc.ycrc.yale.edu): /proc/driver/nvidia/version does not exist
2025-11-13 18:46:06.673924: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


<tf.Tensor: shape=(), dtype=float64, numpy=nan>